In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import joblib

In [2]:
RAW = Path("../Dataset/Raw")
PROCESSED = Path("../Dataset/Processed")

df_meta = pd.read_csv(RAW / "games_detailed_info.csv", low_memory=False)
df_meta = df_meta.rename(columns={"primary":"name","boardgamemechanic":"mechanic","boardgamecategory":"category"})
df_meta = df_meta[["id","name","description"]].drop_duplicates(subset="id").set_index("id")

df_reviews = pd.read_csv(PROCESSED / "reviews_with_sentiment.csv", low_memory=False)

In [3]:
df_meta["description"] = df_meta["description"].fillna("")

tf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
desc_tfidf = tf.fit_transform(df_meta["description"].values)
print("TF-IDF shape:", desc_tfidf.shape)
joblib.dump(tf, "../Dataset/Processed/tfidf_vec.joblib")
joblib.dump(desc_tfidf, "../Dataset/Processed/desc_tfidf.npz")

TF-IDF shape: (21631, 20000)


['../Dataset/Processed/desc_tfidf.npz']

In [4]:
mech = pd.read_csv(RAW / "mechanics.csv", low_memory=False).set_index("BGGId")
themes = pd.read_csv(RAW / "themes.csv", low_memory=False).set_index("BGGId")
subc = pd.read_csv(RAW / "subcategories.csv", low_memory=False).set_index("BGGId")

for df in [mech, themes, subc]:
    df.index = df.index.astype(int)

ids = df_meta.index.astype(int)
mech = mech.reindex(ids).fillna(0)
themes = themes.reindex(ids).fillna(0)
subc = subc.reindex(ids).fillna(0)

content_matrix = np.hstack([mech.values, themes.values, subc.values])
content_matrix = normalize(content_matrix, norm='l2', axis=1)
print("Content matrix shape:", content_matrix.shape)
joblib.dump(content_matrix, "../Dataset/Processed/content_matrix.npy")

Content matrix shape: (21631, 384)


['../Dataset/Processed/content_matrix.npy']

In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')
games_in_reviews = df_reviews["ID"].unique()
print("Games in reviews sample:", len(games_in_reviews))

from collections import defaultdict
game_texts = defaultdict(list)
for _, row in df_reviews.iterrows():
    gid = int(row["ID"])
    if len(game_texts[gid]) < 50:
        game_texts[gid].append(str(row["comment"]))

game_ids = []
embeddings = []
for gid, texts in game_texts.items():
    emb = model.encode(texts, show_progress_bar=False)
    mean_emb = emb.mean(axis=0)
    game_ids.append(gid)
    embeddings.append(mean_emb)

emb_df = pd.DataFrame(embeddings, index=game_ids)
emb_df.index.name = "id"
emb_df.to_parquet("../Dataset/Processed/game_review_embeddings.parquet")
print("Saved per-game embeddings:", emb_df.shape)

Games in reviews sample: 20
Saved per-game embeddings: (20, 384)


In [6]:
import numpy as np
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import joblib

def topk_cosine_sparse(X, k=50, batch_size=512):
    """
    X: matrix [N, D] (can be dense or sparse)
    Return CSR sparse similarity [N, N] with only top-k per row.
    """
    # make sure we can slice rows efficiently
    if sp.issparse(X):
        X = sp.csr_matrix(X)

    N = X.shape[0]
    rows, cols, data = [], [], []

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)

        sims = cosine_similarity(X[start:end], X, dense_output=True)  # [B, N]

        # remove self similarity for rows in this batch
        for i in range(end - start):
            sims[i, start + i] = -1.0

        # take top-k columns per row
        idx = np.argpartition(sims, -k, axis=1)[:, -k:]
        vals = np.take_along_axis(sims, idx, axis=1)

        # sort descending within top-k
        order = np.argsort(-vals, axis=1)
        idx = np.take_along_axis(idx, order, axis=1)
        vals = np.take_along_axis(vals, order, axis=1)

        # write sparse entries
        for i in range(end - start):
            r = start + i
            c = idx[i]
            v = vals[i]

            # keep only positive scores (optional but recommended)
            mask = v > 0
            c = c[mask]
            v = v[mask]

            rows.extend([r] * len(c))
            cols.extend(c.tolist())
            data.extend(v.tolist())

    S = sp.csr_matrix((data, (rows, cols)), shape=(N, N))
    return S


# ---------- DESC (TF-IDF) ----------
desc_tfidf = joblib.load("../Dataset/Processed/desc_tfidf.npz")
desc_topk = topk_cosine_sparse(desc_tfidf, k=50, batch_size=512)
joblib.dump(desc_topk, "../Dataset/Processed/desc_topk_csr.joblib")
print("Saved desc_topk_csr.joblib:", desc_topk.shape, "nnz=", desc_topk.nnz)

# ---------- CONTENT (mechanics+themes+subcategories) ----------
# content_matrix is already created in the previous cell
content_topk = topk_cosine_sparse(content_matrix, k=50, batch_size=256)
joblib.dump(content_topk, "../Dataset/Processed/content_topk_csr.joblib")
print("Saved content_topk_csr.joblib:", content_topk.shape, "nnz=", content_topk.nnz)

# ---------- EMBEDDING (SentenceTransformer) ----------
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors
import joblib

# 1) Load embedding parquet (index = game id)
emb_df = pd.read_parquet("../Dataset/Processed/game_review_embeddings.parquet")
emb_ids = emb_df.index.to_numpy()          # these are BGG ids (like 13, 822, 30549)
emb_vecs = emb_df.values.astype(np.float32)

print("Embedding rows:", emb_vecs.shape[0], "dims:", emb_vecs.shape[1])
print("First 5 embedding ids:", emb_ids[:5])

# 2) Load global id list (21631 items) in the SAME order you use elsewhere
df_meta = (
    pd.read_csv("../Dataset/Raw/games_detailed_info.csv", low_memory=False)
      .rename(columns={"primary": "name"})
      .set_index("id")
)

global_ids = df_meta.index.to_numpy()
N_global = len(global_ids)
id_to_global_idx = {gid: i for i, gid in enumerate(global_ids)}

print("Global items:", N_global)

# 3) Nearest neighbors inside embedding space
N_emb = emb_vecs.shape[0]
k = 50
k_eff = min(k, max(N_emb - 1, 1))
n_neighbors = k_eff + 1
print(f"Using k_eff={k_eff} (n_neighbors={n_neighbors})")

nn = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="auto")
nn.fit(emb_vecs)

dist, ind = nn.kneighbors(emb_vecs, return_distance=True)
sim = 1.0 - dist  # cosine distance -> cosine similarity

# 4) Build CSR BUT in global coordinates (21631x21631)
rows, cols, data = [], [], []

missing_in_global = 0

for i in range(N_emb):
    src_id = int(emb_ids[i])
    if src_id not in id_to_global_idx:
        missing_in_global += 1
        continue

    src_g = id_to_global_idx[src_id]

    for j_local, s in zip(ind[i], sim[i]):
        tgt_id = int(emb_ids[j_local])

        if tgt_id == src_id:
            continue
        if s <= 0:
            continue
        if tgt_id not in id_to_global_idx:
            continue

        tgt_g = id_to_global_idx[tgt_id]

        rows.append(src_g)
        cols.append(tgt_g)
        data.append(float(s))

emb_topk_full = sp.csr_matrix((data, (rows, cols)), shape=(N_global, N_global))

joblib.dump(emb_topk_full, "../Dataset/Processed/emb_topk_csr.joblib")

print("Missing embedding ids not found in global meta:", missing_in_global)
print("Saved emb_topk_csr.joblib (FULL):", emb_topk_full.shape, "nnz=", emb_topk_full.nnz)

print("DONE: Saved TOP-K sparse similarity matrices (no more NxN dense).")

Saved desc_topk_csr.joblib: (21631, 21631) nnz= 1081500
Saved content_topk_csr.joblib: (21631, 21631) nnz= 1057971
Embedding rows: 20 dims: 384
First 5 embedding ids: [    13    822  30549  68448 167791]
Global items: 21631
Using k_eff=19 (n_neighbors=20)
Missing embedding ids not found in global meta: 0
Saved emb_topk_csr.joblib (FULL): (21631, 21631) nnz= 380
DONE: Saved TOP-K sparse similarity matrices (no more NxN dense).


In [7]:
import pandas as pd

emb_df = pd.read_parquet("../Dataset/Processed/game_review_embeddings.parquet")
print("emb_df shape:", emb_df.shape)
print("columns:", emb_df.columns[:10])
print(emb_df.head(3))

emb_df shape: (20, 384)
columns: Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype='int64')
            0         1         2         3         4         5         6    \
id                                                                            
13    -0.005495  0.004273 -0.005156 -0.071926 -0.036092  0.029472  0.025512   
822   -0.010019  0.012368 -0.008759 -0.061605 -0.038494  0.022958  0.028942   
30549 -0.011467 -0.004149 -0.010307 -0.060206 -0.050300  0.038265  0.011772   

            7         8         9    ...       374       375       376  \
id                                   ...                                 
13     0.007183 -0.011964  0.047522  ...  0.092618  0.029714 -0.020307   
822    0.003271 -0.010681  0.047730  ...  0.097177  0.041400 -0.023884   
30549  0.006215 -0.009093  0.045338  ...  0.094833  0.022950 -0.012330   

            377       378       379       380       381       382       383  
id                                                                   